# 02 – Kiểm tra chất lượng dữ liệu (Bước 2)

**Mục tiêu:** Báo cáo đầy đủ các vấn đề chất lượng TRƯỚC khi sửa:
- Thiếu giá trị (NaN + sentinel)
- Trùng timestamp
- Thiếu timestamp (gap)
- Ngoài khoảng vật lý
- Spike
- Flatline

**Nguyên tắc:** KHÔNG sửa dữ liệu trong notebook này.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from parsers import load_all_intermagnet, load_usgs_csv
from quality_checks import run_all_checks
import pickle

print('Setup xong.')

In [ ]:
# Nạp dữ liệu thô (chưa xử lý)
station_dfs = {}
for station in cfg.STATIONS:
    try:
        df, _ = load_all_intermagnet(station)
        station_dfs[station] = df
        print(f'✅ {station}: {len(df):,} dòng')
    except FileNotFoundError as e:
        print(f'⚠  {e}')

In [ ]:
# ── Chạy kiểm tra chất lượng cho từng trạm ────────────────────────────────
qc_results_before = {}  # Lưu để dùng ở Bước 4

for station, df in station_dfs.items():
    result = run_all_checks(df, station=station, label='before')
    qc_results_before[station] = result
    print(f'\n  → Xong {station}')

In [ ]:
# ── Lưu kết quả để dùng cho Bước 3/4/5 ───────────────────────────────────
interim_dir = cfg.DATA_INTERIM
interim_dir.mkdir(parents=True, exist_ok=True)

# Lưu dưới dạng pickle (nhanh và giữ nguyên kiểu dữ liệu)
qc_path = interim_dir / 'qc_before.pkl'
with open(qc_path, 'wb') as f:
    pickle.dump(qc_results_before, f)
print(f'✅ Lưu QC results: {qc_path}')

# Cũng lưu DataFrame thô dưới dạng parquet để dùng ở notebook 03
for station, df in station_dfs.items():
    raw_parquet = interim_dir / f'raw_{station}.parquet'
    df.to_parquet(raw_parquet, index=False)
    print(f'✅ Lưu raw DataFrame: {raw_parquet}')

In [ ]:
# ── In bảng tóm tắt chỉ số chất lượng ────────────────────────────────────
print('\n' + '='*65)
print('  TÓM TẮT BƯỚC 2 – KIỂM TRA CHẤT LƯỢNG (TRƯỚC XỬ LÝ)')
print('='*65)

for station, result in qc_results_before.items():
    print(f'\n[{station}]')
    
    # Missing
    miss = result.get('missing')
    if miss is not None and not miss.empty:
        total_miss = miss['total_missing'].sum()
        print(f'  Thiếu giá trị : {total_miss:,} điểm')
    
    # Duplicates
    dup = result.get('duplicates', {})
    print(f'  Trùng TS      : {dup.get("duplicate_count", 0):,}')
    
    # Gaps
    gap = result.get('gaps', {})
    print(f'  Gap điểm      : {gap.get("missing_points", 0):,} ({gap.get("n_gaps", 0)} gap, '
          f'dài nhất {gap.get("max_gap_minutes", 0)} phút)')
    
    # Bounds
    bnd = result.get('bounds')
    if bnd is not None and not bnd.empty:
        total_out = bnd['out_of_range'].sum()
        print(f'  Ngoài khoảng  : {total_out:,}')
    
    # Spikes
    spk = result.get('spikes')
    if spk is not None and not spk.empty:
        total_spk = spk['spike_count'].sum()
        print(f'  Spike         : {total_spk:,}')
    
    # Flatlines
    flt = result.get('flatlines')
    if flt is not None and not flt.empty:
        total_flt = flt['total_flagged_points'].sum()
        print(f'  Flatline      : {total_flt:,} điểm ({flt["n_segments"].sum()} đoạn)')

print('\n→ Xác nhận để tiếp tục Bước 3 (Xử lý)')